### Real-life Example: Using Splink Results for Machine Learning Model Training

This example will show how to:

1. Use Splink to generate potential matches
2. Manually label some data for supervised learning
3. Train a machine learning model (Logistic Regression) to improve matching accuracy
4. Predict duplicate records using the trained ML model

In [1]:
!pip install splink pandas scikit-learn

  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 285.5 kB/s eta 0:00:0000:0100:02
Using cached joblib-1.4.2-py3-none-any.whl (301 kB)


In [1]:
import json
import pandas as pd
import splink.comparison_library as cl
from splink import SettingsCreator, block_on, Linker, DuckDBAPI
from splink.datasets import splink_datasets, splink_dataset_labels
import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# ========= Load and Prepare Data ========= #
def load_data():
    """Load dataset and remove unnecessary columns."""
    df = splink_datasets.fake_1000.drop(columns=["cluster"], errors="ignore")
    return df

df = load_data()

# ========= Configure Splink Settings ========= #
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("first_name"),
        cl.NameComparison("surname"),
        cl.LevenshteinAtThresholds("dob", 1),
        cl.ExactMatch("city").configure(term_frequency_adjustments=True),
        cl.EmailComparison("email"),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("first_name", "city"),
        block_on("surname"),
    ],
    retain_intermediate_calculation_columns=True,
)

# ========= Generate Record Linkage Predictions ========= #
def generate_predictions(df, settings):
    """Run Splink linkage and generate match probabilities."""
    linker = Linker(df, settings, db_api=DuckDBAPI())
    df_predictions = linker.inference.predict(threshold_match_probability=0.2)
    return df_predictions.as_pandas_dataframe()

df_pred = generate_predictions(df, settings)
df_pred.to_csv("splink_predictions.csv", index=False)

# ========= Load Labeled Data for Training ========= #
def load_labeled_data(sample_size=300):
    """Load labeled dataset and sample records for training."""
    df_labels = splink_dataset_labels.fake_1000_labels
    df_sampled = df_labels.sample(n=min(sample_size, len(df_labels)), random_state=42)
    df_sampled.to_csv("labeled_data.csv", index=False)
    return df_sampled

df_labeled = load_labeled_data()

# ========= Train a Machine Learning Model ========= #
def train_model(df_labeled):
    """Train a logistic regression model on labeled data."""
    if "clerical_match_score" not in df_labeled.columns:
        raise ValueError("Missing required feature: 'clerical_match_score'")

    df_labeled = df_labeled.copy()
    df_labeled.rename(columns={"clerical_match_score": "is_match"}, inplace=True)

    X = df_labeled[["is_match"]]  # Feature for training
    y = df_labeled["is_match"]  # Target variable

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    model = LogisticRegression()
    model.fit(X_train, y_train)

    # Model evaluation
    y_pred = model.predict(X_test)
    print("Model Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    joblib.dump(model, "record_linkage_model.pkl")

train_model(df_labeled)

# ========= Predict Matches on New Data ========= #
def predict_new_data():
    """Use trained ML model to predict new matches."""
    df_new = pd.read_csv("splink_predictions.csv")
    model = joblib.load("record_linkage_model.pkl")

    # Handle column name mismatch
    if "match_probability" in df_new.columns and "is_match" in model.feature_names_in_:
        df_new.rename(columns={"match_probability": "is_match"}, inplace=True)

    if "is_match" not in df_new.columns:
        raise ValueError("Missing required feature: 'is_match' in df_new")

    df_new["predicted_match"] = model.predict(df_new[["is_match"]])

    # Define columns for output
    display_columns = ["first_name_l", "first_name_r", "surname_l", "surname_r", "dob_l", "dob_r", "city_l", "city_r", "email_l", "email_r", "is_match", "predicted_match"]

    # Ensure all required columns exist
    missing_columns = [col for col in display_columns if col not in df_new.columns]
    if missing_columns:
        print(f"Warning: The following columns are missing from df_new: {missing_columns}")

    df_new.to_csv("predicted_matches.csv", index=False)
    return df_new[display_columns].head(20)

predicted_results = predict_new_data()
predicted_results


Blocking time: 0.01 seconds
Predict time: 0.16 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'first_name':
    m values not fully trained
Comparison: 'first_name':
    u values not fully trained
Comparison: 'surname':
    m values not fully trained
Comparison: 'surname':
    u values not fully trained
Comparison: 'dob':
    m values not fully trained
Comparison: 'dob':
    u values not fully trained
Comparison: 'city':
    m values not fully trained
Comparison: 'city':
    u values not fully trained
Comparison: 'email':
    m values not fully trained
Comparison: 'email':
    u values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired behaviour, either: 
 - assign a value for `p

Model Accuracy: 1.0
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00        26
         1.0       1.00      1.00      1.00        34

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



,first_name_l,first_name_r,surname_l,surname_r,dob_l,dob_r,city_l,city_r,email_l,email_r,is_match,predicted_match
0,NaN,Evie,Dean,Dean,2015-03-03,2015-03-03,NaN,Pootsmruth,NaN,evihd56@earris-bailey.net,0.963716,1.0
1,Thomas,Thomas,Gabriel,Gabriel,1976-09-15,1976-09-15,Loodon,London,gabriel.t54@nnichls.info,gabriel.t54@nichols.info,0.998730,1.0
2,Thomas,Thomas,Gabriel,Gabriel,1976-08-15,1976-09-15,NaN,London,NaN,gabriel.t54@nlchois.info,0.960894,1.0
3,Theodore,Theodore,Morris,Morris,1978-08-19,1978-08-19,Birmingham,Birmingham,t.m39@brooks-sawyer.com,t.m39@brooks-sawyer.com,1.000000,1.0
4,Ran,Ryan,Cole,Cole,1988-06-27,1988-05-27,NaN,Bristol,r.cole1@ramirez-anthony.com,r.cole1@ramtrez-anihony.com,0.888493,1.0
5,Osacar,Oscra,Moore,Moore,2016-01-12,2016-01-12,Liverpool,Liverpool,omoore64@randall.com,omoore64@randall.com,0.999990,1.0
6,Oscra,Ocar,Moore,Moore,2016-01-12,2016-01-12,Liverpool,Liverpool,omoore64@randall.com,omoore64@randall.com,0.999935,1.0
7,Miaisie,Maisie,Walsh,Walsh,2005-08-20,2005-08-20,Bimminghar,Birmingham,maiie.walsoh@hodge.com,maisie.walsh@hodge.com,0.237370,0.0
8,NaN,Dylan,Macdonald,Macdonald,1985-10-15,1985-10-15,London,Lodon,dmacdoald@riverb-glass.siz,dmacdonald@rivers-glass.biz,0.610684,1.0
9,Harry,Harry,NaN,May,1987-10-28,1987-10-28,Cardiff,Cardiff,hmay32@richardson-rhodes.com,hmay32@richardson-rhodes.com,0.999993,1.0


### Understanding the Result Format
After running predict_new_data(), the output predicted_matches.csv will contain:

Explanation of Columns

✅ first_name_l, surname_l, dob_l, email_l → Left-side record

✅ first_name_r, surname_r, dob_r, email_r → Right-side record

✅ is_match → Ground truth label (from labeled dataset, 1 = True match, 0 = False match)

✅ predicted_match → Model's prediction (1 = Match, 0 = Non-match)

### Verifying the Model’s Accuracy

Before using the model in production, evaluate its performance:

**Compute Confusion Matrix**

In [4]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# Load predictions
df_results = pd.read_csv("predicted_matches.csv")

# Ensure binary classification labels (convert floats, remove NaNs)
df_results["is_match"] = df_results["is_match"].round().astype(int)
df_results["predicted_match"] = df_results["predicted_match"].round().astype(int)

# Compare ground truth (`is_match`) with model prediction (`predicted_match`)
print(confusion_matrix(df_results["is_match"], df_results["predicted_match"]))
print(classification_report(df_results["is_match"], df_results["predicted_match"]))

[[ 60   4]
 [  0 451]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        64
           1       0.99      1.00      1.00       451

    accuracy                           0.99       515
   macro avg       1.00      0.97      0.98       515
weighted avg       0.99      0.99      0.99       515



✅ True Positives (TP) → Model correctly identified a match

✅ True Negatives (TN) → Model correctly identified a non-match

✅ False Positives (FP) → Model incorrectly identified a match

✅ False Negatives (FN) → Model missed a real match

### Manually Inspect Sample Predictions
👀 Look at False Positives & False Negatives to understand where the model fails.

👉 If too many **false positives**, `increase threshold_match_probability` in Splink.

👉 If too many **false negatives**, `lower threshold_match_probability` or improve ML features.

In [13]:
# Show False Positives (wrongly identified as matches)
df_results[(df_results["is_match"] == 0) & (df_results["predicted_match"] == 1)].head(10)

,match_weight,is_match,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,tf_first_name_l,tf_first_name_r,bf_first_name,...,bf_tf_adj_city,email_l,email_r,gamma_email,tf_email_l,tf_email_r,bf_email,bf_tf_adj_email,match_key,predicted_match
201,-0.133968,0,320,321,Amelia,NaN,-1,0.010830,NaN,1.0,...,1.00000,ameliafletcher22@diaz.com,ameliafletcher22@diaz.com,4,0.005070,0.005070,1024.0,0.182996,1,1
231,-0.094227,0,453,456,Davies,Davies,4,0.004813,0.004813,1024.0,...,1.00000,rd@lewis.com,rd@lewis.com,4,0.003802,0.003802,1024.0,0.243994,1,1
484,-0.084010,0,544,545,Oliver,Olirev,3,0.033694,0.001203,8.0,...,0.00436,oliverjones82@bond.biz,oliverjones82@bond.biz,4,0.003802,0.003802,1024.0,0.243994,1,1
485,-0.084010,0,545,546,Olirev,Oliver,3,0.001203,0.033694,8.0,...,0.00436,oliverjones82@bond.biz,oliverjones82@bond.biz,4,0.003802,0.003802,1024.0,0.243994,1,1


In [11]:
# Show False Negatives (real matches missed)
df_results[(df_results["is_match"] == 1) & (df_results["predicted_match"] == 0)].head(10)

,match_weight,is_match,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,tf_first_name_l,tf_first_name_r,bf_first_name,...,bf_tf_adj_city,email_l,email_r,gamma_email,tf_email_l,tf_email_r,bf_email,bf_tf_adj_email,match_key,predicted_match


If your result is **empty**, that means there are **no false negatives** in your dataset. In other words, your model is not missing any real matches (`is_match == 1`) when predicting (`predicted_match == 0`).

### Using the Model for Future Record Deduplication

In [ ]:
# Load New Unlabeled Data
df_new = pd.read_csv("new_data.csv")  # Replace with your actual dataset

In [ ]:
# Generate Splink Match Probabilities
from splink import Linker

# Use the same Splink settings from training
linker = Linker(df_new, settings, db_api=DuckDBAPI())
df_new_predictions = linker.inference.predict(threshold_match_probability=0.2)
df_new = df_new_predictions.as_pandas_dataframe()

In [ ]:
# Apply the ML Model
import joblib

# Load trained model
model = joblib.load("record_linkage_model.pkl")

# Ensure feature names match
if "match_probability" in df_new.columns:
    df_new.rename(columns={"match_probability": "is_match"}, inplace=True)

# Predict duplicate records
df_new["predicted_match"] = model.predict(df_new[["is_match"]])

# Save results
df_new.to_csv("new_predicted_matches.csv", index=False)

`Once the model has identified matches (predicted_match == 1), you can merge duplicate records in your database:`

In [ ]:
# Use the Predictions for Deduplication
# Keep only unique records by removing predicted duplicates
df_unique = df_new[df_new["predicted_match"] == 0]

# Save cleaned dataset
df_unique.to_csv("deduplicated_data.csv", index=False)

!!! TO BE CONTINUED...